# M7 — E12: CatBoost selector (Kaggle)

**Цель.** Проверить, повышает ли CatBoost итоговый macro **Recall@50** над
принятым M6 fusion (`M1 + M2 + multilingual-E5`). В отличие от локальной
версии, этот ноутбук сам строит E5 candidates на Kaggle GPU и не требует
локальных артефактов M1/M3.


## План ноутбука

1. Установить недостающие библиотеки и подключить live ClearML.
2. Воссоздать frozen M0 proxy и category partition.
3. Построить/cache M1, zero-shot E5 и leakage-safe OOF M2 top-200 candidates.
4. Обучить CatBoostRanker только на OOF pool proxy-train.
5. Измерить M6 и CatBoost на untouched validation groups.
6. Сохранить модель, candidate cache, метрики, validation pool и ZIP.


## 1. Установка зависимостей

In [ ]:
!pip install -q "clearml>=1.16" "catboost>=1.2" "sentence-transformers>=3.4" "transformers>=4.51" "snowballstemmer>=2.2"

## 2. Kaggle Input, конфигурация и live ClearML

In [ ]:

from __future__ import annotations

import gc
import importlib.metadata
import json
import os
import re
import sys
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import snowballstemmer
from catboost import CatBoostRanker, Pool
from clearml import Task
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupKFold

SEED = 42
SOURCE_TOP_K = 200
FINAL_K = 50
HISTORY_QUOTA = 20
E5_QUOTA = 10
MAX_POOL_CANDIDATES = 160
MAX_TRAIN_CANDIDATES = 120
OOF_FOLDS = 5
RRF_K = 60
BM25_K1, BM25_B = 1.5, 0.75
CHAR_MAX_FEATURES = 200_000
CHAR_BATCH_SIZE = 32
USE_ALL_VISIBLE_GPUS_FOR_E5 = True
USE_CATBOOST_GPU = False  # CPU is sufficient and more portable for this pool size.
NOTEBOOK_STARTED = time.perf_counter()
np.random.seed(SEED)

os.environ.setdefault("HF_HOME", "/kaggle/temp/hf-cache")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.pop("CLEARML_OFFLINE_MODE", None)
OUTPUT_DIR = Path("/kaggle/working/m7_e12_catboost_selector")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_required_secret(name: str) -> str:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Add Kaggle Secret {name!r} before running this notebook.") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty.")
    return value

def get_optional_secret(name: str) -> None:
    try:
        value = secrets.get_secret(name)
    except Exception:
        return
    if value:
        os.environ[name] = value

os.environ["CLEARML_API_ACCESS_KEY"] = get_required_secret("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = get_required_secret("CLEARML_API_SECRET_KEY")
for _name in ("CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
    get_optional_secret(_name)

INPUT_ROOT = Path("/kaggle/input")
def find_input_file(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.glob(f"**/{filename}"))
    if len(matches) != 1:
        raise RuntimeError(f"Attach exactly one Kaggle input with {filename!r}; found {[str(x) for x in matches]}")
    return matches[0]

TRAIN_PATH = find_input_file("train.parquet")
BENCHMARK_QUERIES_PATH = find_input_file("benchmark_queries.parquet")
BENCHMARK_ITEMS_PATH = find_input_file("benchmark_items.parquet")

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="E12__catboost_selector__m1_m2_e5__kaggle__s42",
    reuse_last_task_id=False,
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_task.connect({
    "stage": "M7_E12_catboost_candidate_selector",
    "environment": "kaggle",
    "validation_protocol": "benchmark_aligned_proxy_v1__group_disjoint__OOF_history",
    "seed": SEED,
    "source_top_k": SOURCE_TOP_K,
    "final_k": FINAL_K,
    "max_pool_candidates": MAX_POOL_CANDIDATES,
    "max_train_candidates": MAX_TRAIN_CANDIDATES,
    "oof_folds": OOF_FOLDS,
    "history_quota": HISTORY_QUOTA,
    "e5_quota": E5_QUOTA,
    "catboost_gpu_requested": USE_CATBOOST_GPU,
}, name="config")
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "output_dir": str(OUTPUT_DIR), "train": str(TRAIN_PATH)})


## 3. Shared dense utilities и frozen M0 proxy

In [ ]:

# Exact local M3 utility snapshot embedded for Kaggle reproducibility.
shared_module = types.ModuleType("dense_retrieval")
exec('"""Shared, exact dense-retrieval utilities for M3 notebooks.\n\nThe module deliberately has no ClearML dependency.  Notebooks own experiment\ntracking; this module owns the frozen proxy, text construction and exact\ncategory-partitioned search so local and Kaggle runs cannot silently diverge.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\nSEARCH_COLUMNS = [\n    "search_query",\n    "search_location_id",\n    "search_is_delivery_search",\n    "search_infm_params_text",\n    "search_category",\n]\nTRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]\nBENCHMARK_QUERY_COLUMNS = ["query_id", *SEARCH_COLUMNS]\nITEM_COLUMNS = [\n    "item_id",\n    "item_title_raw",\n    "item_description_raw",\n    "item_infm_params_text",\n    "item_category_id",\n]\nMETRIC_KS = (1, 5, 10, 20, 50, 200)\n\n# The variants differ by training data, architecture and prompting strategy.\n# All are open-weight, local models; public leaderboards only define a shortlist.\nMODEL_SPECS: dict[str, dict[str, Any]] = {\n    "multilingual_e5_large_instruct": {\n        "model_id": "intfloat/multilingual-e5-large-instruct",\n        "query_mode": "e5_instruction",\n        "query_instruction": "Given a Russian service-search request, retrieve relevant service listings.",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "bge_m3": {\n        "model_id": "BAAI/bge-m3",\n        "query_mode": "plain",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "user_bge_m3": {\n        "model_id": "deepvk/USER-bge-m3",\n        "query_mode": "plain",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "ru_en_rosberta": {\n        "model_id": "ai-forever/ru-en-RoSBERTa",\n        "query_mode": "prefix",\n        "query_prefix": "search_query: ",\n        "document_prefix": "search_document: ",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "qwen3_embedding_0_6b": {\n        "model_id": "Qwen/Qwen3-Embedding-0.6B",\n        "query_mode": "sentence_transformers_prompt",\n        "query_prompt_name": "query",\n        "max_length": 256,\n        "batch_size": 16,\n    },\n}\n\n\ndef canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:\n    """Canonicalize only fields used to identify a logical query group."""\n    result = frame[SEARCH_COLUMNS].copy()\n    for column in ("search_query", "search_infm_params_text"):\n        result[column] = (\n            result[column]\n            .astype("string")\n            .fillna("<NA>")\n            .str.lower()\n            .str.strip()\n            .str.replace(r"\\s+", " ", regex=True)\n        )\n    for column in ("search_location_id", "search_is_delivery_search", "search_category"):\n        result[column] = result[column].astype("string").fillna("<NA>")\n    return result\n\n\ndef load_frozen_proxy(\n    train_path: Path,\n    benchmark_queries_path: Path,\n    benchmark_items_path: Path,\n    *,\n    seed: int,\n) -> dict[str, Any]:\n    """Recreate the M0 benchmark-aligned, group-disjoint validation proxy."""\n    train_pairs = pd.read_parquet(train_path, columns=TRAIN_COLUMNS)\n    benchmark_queries = pd.read_parquet(benchmark_queries_path, columns=BENCHMARK_QUERY_COLUMNS)\n    candidate_items = pd.read_parquet(benchmark_items_path, columns=ITEM_COLUMNS).reset_index(drop=True)\n\n    all_contexts = pd.concat(\n        [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],\n        ignore_index=True,\n    )\n    group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)\n    train_pairs["query_group"] = group_ids[: len(train_pairs)]\n\n    candidate_item_ids = candidate_items["item_id"].astype(str).to_numpy()\n    candidate_item_id_set = set(candidate_item_ids)\n    proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(candidate_item_id_set)].copy()\n\n    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)\n    train_idx, valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))\n    proxy_train_pairs = proxy_pairs.iloc[train_idx].copy()\n    proxy_valid_pairs = proxy_pairs.iloc[valid_idx].copy()\n    assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))\n\n    validation_queries = (\n        proxy_valid_pairs.sort_values("query_group")\n        .drop_duplicates("query_group")\n        .loc[:, ["query_group", *SEARCH_COLUMNS]]\n        .reset_index(drop=True)\n    )\n    gold_by_group = (\n        proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]\n        .agg(lambda values: frozenset(values.astype(str)))\n        .to_dict()\n    )\n    gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]\n\n    category_to_indices = {\n        str(category): group.index.to_numpy(dtype=np.int64)\n        for category, group in candidate_items.groupby("item_category_id", sort=False)\n    }\n    all_indices = np.arange(len(candidate_items), dtype=np.int64)\n    allowed_indices_by_query = [\n        category_to_indices.get(str(category), all_indices)\n        for category in validation_queries["search_category"]\n    ]\n    category_oracle = float(\n        np.mean(\n            [\n                len(gold & set(candidate_item_ids[allowed])) / len(gold)\n                for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)\n            ]\n        )\n    )\n    assert category_oracle == 1.0, "The M0 category partition removed a validation positive."\n\n    return {\n        "train_pairs": train_pairs,\n        "proxy_train_pairs": proxy_train_pairs,\n        "proxy_valid_pairs": proxy_valid_pairs,\n        "validation_queries": validation_queries,\n        "gold_sets": gold_sets,\n        "candidate_items": candidate_items,\n        "candidate_item_ids": candidate_item_ids,\n        "category_to_indices": category_to_indices,\n        "all_indices": all_indices,\n        "category_oracle": category_oracle,\n        "benchmark_queries": benchmark_queries,\n    }\n\n\ndef compose_query_text(frame: pd.DataFrame) -> list[str]:\n    """Keep the request primary; filters are useful but must not replace it."""\n    query = frame["search_query"].fillna("").astype(str).str.strip()\n    filters = frame["search_infm_params_text"].fillna("").astype(str).str.strip()\n    return [\n        value if not params else f"{value}\\nФильтры поиска: {params}"\n        for value, params in zip(query, filters, strict=True)\n    ]\n\n\ndef compose_item_text(frame: pd.DataFrame, *, description_char_limit: int = 1_500) -> list[str]:\n    """Use only user-visible item text; cap descriptions before tokenization."""\n    title = frame["item_title_raw"].fillna("").astype(str).str.strip()\n    params = frame["item_infm_params_text"].fillna("").astype(str).str.strip()\n    description = frame["item_description_raw"].fillna("").astype(str).str.slice(stop=description_char_limit).str.strip()\n    texts: list[str] = []\n    for title_value, params_value, description_value in zip(title, params, description, strict=True):\n        fields = [f"Название услуги: {title_value}"]\n        if params_value:\n            fields.append(f"Параметры: {params_value}")\n        if description_value:\n            fields.append(f"Описание: {description_value}")\n        texts.append("\\n".join(fields))\n    return texts\n\n\ndef prepare_model_texts(texts: list[str], spec: dict[str, Any], *, is_query: bool) -> tuple[list[str], dict[str, Any]]:\n    """Apply only the prompt format declared by each model\'s model card."""\n    if not is_query:\n        prefix = str(spec.get("document_prefix", ""))\n        return [prefix + text for text in texts], {}\n\n    mode = spec["query_mode"]\n    if mode == "plain":\n        return texts, {}\n    if mode == "e5_instruction":\n        instruction = str(spec["query_instruction"])\n        return [f"Instruct: {instruction}\\nQuery: {text}" for text in texts], {}\n    if mode == "prefix":\n        return [str(spec["query_prefix"]) + text for text in texts], {}\n    if mode == "sentence_transformers_prompt":\n        return texts, {"prompt_name": str(spec["query_prompt_name"])}\n    raise ValueError(f"Unknown query_mode={mode!r}")\n\n\ndef load_encoder(spec: dict[str, Any], *, device: str) -> Any:\n    """Load a SentenceTransformer in inference mode without remote inference APIs."""\n    from sentence_transformers import SentenceTransformer\n\n    model = SentenceTransformer(str(spec["model_id"]), device=device)\n    model.max_seq_length = int(spec["max_length"])\n    if device == "cuda":\n        model.half()\n    return model\n\n\ndef model_revision(model: Any) -> str:\n    """Best-effort resolved Hugging Face commit, without relying on a cache path."""\n    candidates = [model]\n    try:\n        candidates.append(model._first_module())\n    except (AttributeError, TypeError):\n        pass\n    for candidate in list(candidates):\n        auto_model = getattr(candidate, "auto_model", None)\n        if auto_model is not None:\n            candidates.append(auto_model)\n    for candidate in candidates:\n        config = getattr(candidate, "config", None)\n        revision = getattr(config, "_commit_hash", None)\n        if revision:\n            return str(revision)\n    return "unavailable"\n\n\ndef encode_texts(\n    model: Any,\n    spec: dict[str, Any],\n    texts: list[str],\n    *,\n    is_query: bool,\n    device: str,\n    pool: Any | None = None,\n) -> np.ndarray:\n    """Encode texts on one device or a reusable SentenceTransformer GPU pool."""\n    prepared, extra_kwargs = prepare_model_texts(texts, spec, is_query=is_query)\n    batch_size = int(spec["batch_size"] if device == "cuda" else min(8, int(spec["batch_size"])))\n    if pool is not None:\n        # 1,000 texts per inter-process task keeps IPC bounded for 189k items.\n        extra_kwargs.update({"pool": pool, "chunk_size": 1_000})\n    vectors = model.encode(\n        prepared,\n        batch_size=batch_size,\n        show_progress_bar=True,\n        convert_to_numpy=True,\n        normalize_embeddings=True,\n        **extra_kwargs,\n    )\n    return np.asarray(vectors, dtype=np.float32)\n\n\ndef exact_category_search(\n    query_embeddings: np.ndarray,\n    item_embeddings: np.ndarray,\n    query_categories: list[object],\n    category_to_indices: dict[str, np.ndarray],\n    item_ids: np.ndarray,\n    *,\n    top_k: int,\n    query_batch_size: int = 128,\n) -> tuple[list[list[str]], int]:\n    """Exact cosine/IP retrieval inside M0 category partitions.\n\n    Embeddings are L2-normalized.  The function uses a bounded query batch so\n    it does not allocate the full query-by-corpus score matrix.\n    """\n    if query_embeddings.shape[1] != item_embeddings.shape[1]:\n        raise ValueError("Query and item embedding dimensions differ.")\n    rankings: list[list[str]] = [[] for _ in range(len(query_embeddings))]\n    all_indices = np.arange(len(item_ids), dtype=np.int64)\n    fallback_count = 0\n    positions_by_category: dict[str, list[int]] = {}\n    for position, category in enumerate(query_categories):\n        positions_by_category.setdefault(str(category), []).append(position)\n\n    for category, positions in positions_by_category.items():\n        allowed = category_to_indices.get(category)\n        if allowed is None or len(allowed) == 0:\n            allowed = all_indices\n            fallback_count += len(positions)\n        k = min(top_k, len(allowed))\n        if k == 0:\n            continue\n        for start in range(0, len(positions), query_batch_size):\n            batch_positions = positions[start : start + query_batch_size]\n            scores = query_embeddings[batch_positions] @ item_embeddings[allowed].T\n            top_local = np.argpartition(scores, kth=scores.shape[1] - k, axis=1)[:, -k:]\n            top_scores = np.take_along_axis(scores, top_local, axis=1)\n            order = np.argsort(top_scores, axis=1)[:, ::-1]\n            top_global = allowed[np.take_along_axis(top_local, order, axis=1)]\n            for query_position, item_positions in zip(batch_positions, top_global, strict=True):\n                rankings[query_position] = item_ids[item_positions].astype(str).tolist()\n    return rankings, fallback_count\n\n\ndef macro_recall_at_k(rankings: list[list[str]], gold_sets: list[frozenset[str]], k: int) -> float:\n    return float(\n        np.mean(\n            [\n                len(set(prediction[:k]) & relevant) / len(relevant)\n                for prediction, relevant in zip(rankings, gold_sets, strict=True)\n            ]\n        )\n    )\n\n\ndef hit_rate_at_k(rankings: list[list[str]], gold_sets: list[frozenset[str]], k: int) -> float:\n    return float(\n        np.mean(\n            [\n                bool(set(prediction[:k]) & relevant)\n                for prediction, relevant in zip(rankings, gold_sets, strict=True)\n            ]\n        )\n    )\n\n\ndef evaluate_rankings(\n    rankings: list[list[str]],\n    gold_sets: list[frozenset[str]],\n    *,\n    model_name: str,\n    model_id: str,\n    item_encoding_seconds: float,\n    query_encoding_seconds: float,\n    search_seconds: float,\n    embedding_dimension: int,\n    category_fallback_queries: int,\n) -> dict[str, Any]:\n    record: dict[str, Any] = {\n        "model": model_name,\n        "model_id": model_id,\n        "embedding_dimension": int(embedding_dimension),\n        "item_encoding_seconds": float(item_encoding_seconds),\n        "query_encoding_seconds": float(query_encoding_seconds),\n        "search_seconds": float(search_seconds),\n        "category_fallback_queries": int(category_fallback_queries),\n        "mean_candidates": float(np.mean([len(row) for row in rankings])),\n    }\n    for k in METRIC_KS:\n        record[f"recall@{k}"] = macro_recall_at_k(rankings, gold_sets, k)\n    record["hit_rate@50"] = hit_rate_at_k(rankings, gold_sets, 50)\n    return record\n\n\ndef save_rankings_jsonl(path: Path, query_groups: pd.Series, rankings: list[list[str]]) -> None:\n    """Persist validation candidates for later union/error analysis, not an index."""\n    with path.open("w", encoding="utf-8") as handle:\n        for query_group, item_ids in zip(query_groups, rankings, strict=True):\n            handle.write(\n                json.dumps(\n                    {"query_group": int(query_group), "candidate_item_ids": item_ids},\n                    ensure_ascii=False,\n                )\n                + "\\n"\n            )\n\n\ndef hardware_snapshot() -> dict[str, Any]:\n    """Return safe hardware facts for ClearML and the reproducibility manifest."""\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            properties = torch.cuda.get_device_properties(0)\n            return {\n                "device": "cuda",\n                "gpu_name": properties.name,\n                "gpu_total_memory_gb": round(properties.total_memory / 1024**3, 2),\n            }\n    except ImportError:\n        pass\n    return {"device": "cpu"}\n', shared_module.__dict__)
sys.modules["dense_retrieval"] = shared_module
from dense_retrieval import (
    MODEL_SPECS, compose_item_text, compose_query_text, encode_texts,
    exact_category_search, hardware_snapshot, load_encoder, load_frozen_proxy,
)


In [ ]:

SEARCH_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
ITEM_FEATURE_COLUMNS = [
    "item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
    "item_category_id", "item_price", "item_rating", "item_rating_reviews_count",
    "item_location_id", "item_is_phone_hidden", "item_is_message_forbidden",
]

load_started = time.perf_counter()
proxy = load_frozen_proxy(TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH, seed=SEED)
train_pairs = proxy["train_pairs"].copy()
proxy_train_pairs = proxy["proxy_train_pairs"].copy()
proxy_valid_pairs = proxy["proxy_valid_pairs"].copy()
validation_queries = proxy["validation_queries"].copy()
candidate_items = proxy["candidate_items"].copy()
item_features = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_FEATURE_COLUMNS)
assert np.array_equal(item_features["item_id"].astype(str).to_numpy(), candidate_items["item_id"].astype(str).to_numpy())

NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")
STEMMER = snowballstemmer.stemmer("russian")
def normalize_russian_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    return " ".join(NON_WORD_RE.sub(" ", text.lower().replace("ё", "е")).split())
def stem_russian_text(value: object) -> str:
    return " ".join(STEMMER.stemWords(normalize_russian_text(value).split()))

# M1 E04's retained representations are regenerated from the three attached
# source files. This removes a dependency on a local materialized Parquet.
m1_items = candidate_items.loc[:, ["item_id", "item_category_id"]].copy()
full_item_text = (
    candidate_items["item_title_raw"].fillna("").astype(str) + " "
    + candidate_items["item_infm_params_text"].fillna("").astype(str) + " "
    + candidate_items["item_description_raw"].fillna("").astype(str)
)
m1_items["item_text_bm25"] = full_item_text.map(stem_russian_text)
m1_items["item_title_char_tfidf"] = candidate_items["item_title_raw"].map(normalize_russian_text)

for frame in (train_pairs, proxy_train_pairs, proxy_valid_pairs, validation_queries):
    frame["query_text_norm"] = frame["search_query"].map(normalize_russian_text)
    frame["category_key"] = frame["search_category"].astype(str)

proxy_train_queries = (
    proxy_train_pairs.sort_values("query_group").drop_duplicates("query_group")
    [["query_group", *SEARCH_COLUMNS, "query_text_norm", "category_key"]].reset_index(drop=True)
)
validation_queries = validation_queries.sort_values("query_group").reset_index(drop=True)
all_selector_queries = pd.concat([proxy_train_queries, validation_queries], ignore_index=True)
assert all_selector_queries["query_group"].is_unique

gold_by_group = proxy_valid_pairs.groupby("query_group", sort=False)["item_id"].agg(
    lambda values: frozenset(values.astype(str))
).to_dict()
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]
item_ids = candidate_items["item_id"].astype(str).to_numpy()
item_id_set = set(item_ids)
category_to_indices = {
    str(category): group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}
all_indices = np.arange(len(candidate_items), dtype=np.int64)
category_to_item_ids = {
    str(category): set(group["item_id"].astype(str))
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}
print({
    "load_seconds": round(time.perf_counter() - load_started, 2),
    "proxy_train_groups": len(proxy_train_queries),
    "validation_groups": len(validation_queries),
    "candidate_items": len(candidate_items),
    "category_oracle_recall": proxy["category_oracle"],
})


## 4. M1 lexical candidates и multilingual-E5 GPU cache

In [ ]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"

class SparseBM25:
    """Exact retained M1 BM25 implementation; only source ranks are exported."""

    def __init__(self, k1: float = BM25_K1, b: float = BM25_B, epsilon: float = 0.25):
        self.k1, self.b, self.epsilon = k1, b, epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, dtype=np.float32)
        counts = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.matrix = counts.tocsc()
        self.n_docs = counts.shape[0]
        df = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - df + 0.5) / (df + 0.5))
        idf[idf < 0] = self.epsilon * float(idf.mean())
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.doc_len.mean())
        self.vocabulary = self.vectorizer.vocabulary_
        return self

    def scores(self, query: str) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(query.split()).items():
            feature = self.vocabulary.get(token)
            if feature is None:
                continue
            start, stop = self.matrix.indptr[feature : feature + 2]
            rows, term_tf = self.matrix.indices[start:stop], self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature] * term_tf * (self.k1 + 1) / (term_tf + self.norm[rows])
        return scores


def top_k_indices(scores: np.ndarray, allowed: np.ndarray, k: int = SOURCE_TOP_K) -> np.ndarray:
    k = min(k, len(allowed))
    allowed_scores = scores[allowed]
    selected = np.argpartition(allowed_scores, len(allowed) - k)[len(allowed) - k:]
    return allowed[selected[np.argsort(allowed_scores[selected])[::-1]]]


def rrf_fuse(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    scores: dict[int, float] = {}
    for ranking in (left, right):
        for rank, item_idx in enumerate(ranking, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (RRF_K + rank)
    return np.asarray(sorted(scores, key=lambda idx: (-scores[idx], idx))[:SOURCE_TOP_K], dtype=np.int64)


def write_source_cache(path: Path, rows: list[dict[str, object]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_source_cache(path: Path) -> dict[int, dict[str, list[str]]]:
    parsed: dict[int, dict[str, list[str]]] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        row = json.loads(raw_line)
        parsed[int(row["query_group"])] = {
            key: [str(item_id) for item_id in row[key]]
            for key in ("m1", "bm25", "char")
        }
    return parsed


M1_CACHE_PATH = CACHE_DIR / "m1_top200_all_selector_queries.jsonl"
expected_groups = set(map(int, all_selector_queries["query_group"]))
if M1_CACHE_PATH.exists():
    m1_sources_by_group = read_source_cache(M1_CACHE_PATH)
    assert set(m1_sources_by_group) == expected_groups, "Delete stale M1 cache before changing proxy/config."
    print({"m1_cache": "hit", "path": str(M1_CACHE_PATH)})
else:
    m1_started = time.perf_counter()
    bm25 = SparseBM25().fit(m1_items["item_text_bm25"].fillna("").astype(str).tolist())
    bm25_top: list[np.ndarray] = []
    for query, category in zip(all_selector_queries["search_query"], all_selector_queries["category_key"], strict=True):
        allowed = category_to_indices.get(str(category), all_indices)
        bm25_top.append(top_k_indices(bm25.scores(stem_russian_text(query)), allowed))
    del bm25
    gc.collect()

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb", lowercase=False, ngram_range=(3, 5), min_df=2,
        max_features=CHAR_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
    )
    char_matrix = char_vectorizer.fit_transform(m1_items["item_title_char_tfidf"].fillna("").astype(str))
    query_matrix = char_vectorizer.transform(all_selector_queries["search_query"].map(normalize_russian_text).tolist())
    char_top: list[np.ndarray] = []
    for start in range(0, query_matrix.shape[0], CHAR_BATCH_SIZE):
        stop = min(start + CHAR_BATCH_SIZE, query_matrix.shape[0])
        scores_batch = (query_matrix[start:stop] @ char_matrix.T).toarray()
        for scores, category in zip(scores_batch, all_selector_queries["category_key"].iloc[start:stop], strict=True):
            allowed = category_to_indices.get(str(category), all_indices)
            char_top.append(top_k_indices(scores, allowed))
    del char_matrix, query_matrix, char_vectorizer
    gc.collect()

    cache_rows: list[dict[str, object]] = []
    for group, bm25_rank, char_rank in zip(all_selector_queries["query_group"], bm25_top, char_top, strict=True):
        cache_rows.append({
            "query_group": int(group),
            "m1": item_ids[rrf_fuse(bm25_rank, char_rank)].astype(str).tolist(),
            "bm25": item_ids[bm25_rank].astype(str).tolist(),
            "char": item_ids[char_rank].astype(str).tolist(),
        })
    write_source_cache(M1_CACHE_PATH, cache_rows)
    m1_sources_by_group = read_source_cache(M1_CACHE_PATH)
    print({"m1_cache": "built", "seconds": round(time.perf_counter() - m1_started, 2), "path": str(M1_CACHE_PATH)})


def read_dense_cache(path: Path) -> dict[int, list[str]]:
    return {
        int(row["query_group"]): [str(item_id) for item_id in row["candidate_item_ids"]]
        for row in (json.loads(line) for line in path.read_text(encoding="utf-8").splitlines())
    }


def write_dense_cache(path: Path, groups: pd.Series, rankings: list[list[str]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for group, ranking in zip(groups, rankings, strict=True):
            handle.write(json.dumps({"query_group": int(group), "candidate_item_ids": ranking}, ensure_ascii=False) + "\n")



DENSE_ALL_CACHE_PATH = CACHE_DIR / "e5_proxy_train_and_validation_top200.jsonl"
expected_groups = set(map(int, all_selector_queries["query_group"]))
if DENSE_ALL_CACHE_PATH.exists():
    e5_sources_by_group = read_dense_cache(DENSE_ALL_CACHE_PATH)
    assert set(e5_sources_by_group) == expected_groups, "Delete stale E5 cache before changing proxy/config."
    print({"e5_cache": "hit", "path": str(DENSE_ALL_CACHE_PATH)})
else:
    import torch

    hardware = hardware_snapshot()
    if hardware["device"] != "cuda":
        raise RuntimeError("Enable a Kaggle GPU accelerator before running M7.")
    visible_devices = [f"cuda:{index}" for index in range(torch.cuda.device_count())]
    assert visible_devices, "No CUDA device is visible to PyTorch."
    encoding_devices = visible_devices if USE_ALL_VISIBLE_GPUS_FOR_E5 else visible_devices[:1]
    use_pool = len(encoding_devices) > 1
    spec = MODEL_SPECS["multilingual_e5_large_instruct"]
    model = load_encoder(spec, device="cpu" if use_pool else "cuda")
    pool = None
    item_embeddings = query_embeddings = None
    dense_started = time.perf_counter()
    try:
        if use_pool:
            pool = model.start_multi_process_pool(target_devices=encoding_devices)
        item_embeddings = encode_texts(
            model, spec, compose_item_text(candidate_items), is_query=False, device="cuda", pool=pool,
        )
        query_embeddings = encode_texts(
            model, spec, compose_query_text(all_selector_queries), is_query=True, device="cuda", pool=pool,
        )
        rankings, fallback_count = exact_category_search(
            query_embeddings, item_embeddings, all_selector_queries["search_category"].tolist(),
            category_to_indices, item_ids, top_k=SOURCE_TOP_K,
        )
        assert fallback_count >= 0
        write_dense_cache(DENSE_ALL_CACHE_PATH, all_selector_queries["query_group"], rankings)
    finally:
        if pool is not None:
            model.stop_multi_process_pool(pool)
        del model, item_embeddings, query_embeddings
        gc.collect()
        torch.cuda.empty_cache()
    e5_sources_by_group = read_dense_cache(DENSE_ALL_CACHE_PATH)
    print({
        "e5_cache": "built", "seconds": round(time.perf_counter() - dense_started, 2),
        "encoding_devices": encoding_devices, "path": str(DENSE_ALL_CACHE_PATH),
    })
assert set(e5_sources_by_group) == expected_groups


## 5. Leakage-safe OOF M2 history candidates

In [ ]:
def build_history_lookup(frame: pd.DataFrame) -> dict[str, list[str]]:
    unique = frame[["query_group", "query_text_norm", "item_id"]].drop_duplicates()
    counts = unique.groupby(["query_text_norm", "item_id"], sort=False).size().rename("support").reset_index()
    lookup: dict[str, list[str]] = {}
    for query_text, part in counts.groupby("query_text_norm", sort=False):
        part = part.sort_values(["support", "item_id"], ascending=[False, True])
        lookup[str(query_text)] = part["item_id"].astype(str).tolist()
    return lookup


def filter_history_by_category(candidates: list[str], category_key: str) -> list[str]:
    allowed = category_to_item_ids.get(str(category_key), item_id_set)
    return [item_id for item_id in candidates if item_id in allowed][:SOURCE_TOP_K]


def nearest_history_rankings(fit_pairs: pd.DataFrame, target_queries: pd.DataFrame) -> dict[int, list[str]]:
    lookup = build_history_lookup(fit_pairs)
    history_texts = np.asarray(sorted(lookup), dtype=object)
    if len(history_texts) == 0:
        return {int(group): [] for group in target_queries["query_group"]}
    vectorizer = TfidfVectorizer(
        analyzer="char_wb", preprocessor=normalize_russian_text, lowercase=False,
        ngram_range=(3, 5), min_df=1, max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True, dtype=np.float32,
    )
    history_matrix = vectorizer.fit_transform(history_texts)
    target_matrix = vectorizer.transform(target_queries["query_text_norm"].tolist())
    result: dict[int, list[str]] = {}
    for start in range(0, target_matrix.shape[0], CHAR_BATCH_SIZE):
        stop = min(start + CHAR_BATCH_SIZE, target_matrix.shape[0])
        scores_batch = (target_matrix[start:stop] @ history_matrix.T).toarray()
        for group, category, scores in zip(
            target_queries["query_group"].iloc[start:stop],
            target_queries["category_key"].iloc[start:stop],
            scores_batch,
            strict=True,
        ):
            best_idx = int(np.argmax(scores))
            result[int(group)] = (
                filter_history_by_category(lookup[str(history_texts[best_idx])], str(category))
                if float(scores[best_idx]) > 0 else []
            )
    del history_matrix, target_matrix, vectorizer
    return result


HISTORY_CACHE_PATH = CACHE_DIR / "m2_oof_and_validation_top200.jsonl"
if HISTORY_CACHE_PATH.exists():
    history_sources_by_group = {
        int(row["query_group"]): [str(item_id) for item_id in row["candidate_item_ids"]]
        for row in (json.loads(line) for line in HISTORY_CACHE_PATH.read_text(encoding="utf-8").splitlines())
    }
    assert set(history_sources_by_group) == expected_groups, "Delete stale history cache before changing proxy/config."
    print({"history_cache": "hit", "path": str(HISTORY_CACHE_PATH)})
else:
    history_started = time.perf_counter()
    history_sources_by_group: dict[int, list[str]] = {}
    group_values = proxy_train_queries["query_group"].to_numpy()
    splitter = GroupKFold(n_splits=OOF_FOLDS)
    for fold, (_, target_positions) in enumerate(splitter.split(proxy_train_queries, groups=group_values), start=1):
        target_queries = proxy_train_queries.iloc[target_positions].copy()
        target_groups = set(target_queries["query_group"])
        fit_pairs = proxy_train_pairs.loc[~proxy_train_pairs["query_group"].isin(target_groups)].copy()
        assert target_groups.isdisjoint(set(fit_pairs["query_group"])), "OOF history leakage."
        fold_rankings = nearest_history_rankings(fit_pairs, target_queries)
        history_sources_by_group.update(fold_rankings)
        print({"oof_fold": fold, "target_groups": len(target_groups), "fit_groups": fit_pairs["query_group"].nunique()})
    validation_history = nearest_history_rankings(proxy_train_pairs, validation_queries)
    history_sources_by_group.update(validation_history)
    assert set(history_sources_by_group) == expected_groups
    with HISTORY_CACHE_PATH.open("w", encoding="utf-8") as handle:
        for group in sorted(history_sources_by_group):
            handle.write(json.dumps({"query_group": int(group), "candidate_item_ids": history_sources_by_group[group]}, ensure_ascii=False) + "\n")
    print({"history_cache": "built", "seconds": round(time.perf_counter() - history_started, 2), "path": str(HISTORY_CACHE_PATH)})

## 6. Expanded pool, CatBoostRanker и Recall@50

In [ ]:
def append_unique(target: list[str], source: list[str], limit: int | None = None, max_size: int | None = None) -> None:
    for item_id in source if limit is None else source[:limit]:
        if item_id not in target:
            target.append(item_id)
        if max_size is not None and len(target) >= max_size:
            return


def m6_baseline(m1: list[str], history: list[str], dense: list[str]) -> list[str]:
    selected: list[str] = []
    append_unique(selected, history, HISTORY_QUOTA, FINAL_K)
    append_unique(selected, dense, E5_QUOTA, FINAL_K)
    append_unique(selected, m1, None, FINAL_K)
    return selected[:FINAL_K]


def expanded_pool(m1: list[str], history: list[str], dense: list[str], limit: int) -> tuple[list[str], list[str]]:
    baseline = m6_baseline(m1, history, dense)
    selected = list(baseline)
    for source in (m1, history, dense):
        append_unique(selected, source, SOURCE_TOP_K, limit)
    return selected[:limit], baseline


def rank_lookup(values: list[str]) -> dict[str, int]:
    return {item_id: rank for rank, item_id in enumerate(values, start=1)}


def safe_float(value: object, default: float = 0.0) -> float:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return default
    return result if np.isfinite(result) else default


def token_set(value: object) -> set[str]:
    return set(normalize_russian_text(value).split())


item_feature_map = item_features.assign(item_id=item_features["item_id"].astype(str)).set_index("item_id").to_dict("index")


def build_feature_rows(
    queries: pd.DataFrame,
    *,
    labels_by_group: dict[int, frozenset[str]] | None,
    max_candidates: int,
) -> tuple[pd.DataFrame, dict[int, dict[str, object]]]:
    rows: list[dict[str, object]] = []
    pools: dict[int, dict[str, object]] = {}
    for query in queries.itertuples(index=False):
        group = int(query.query_group)
        m1 = m1_sources_by_group[group]["m1"]
        bm25 = m1_sources_by_group[group]["bm25"]
        char = m1_sources_by_group[group]["char"]
        history = history_sources_by_group[group]
        dense = e5_sources_by_group[group]
        candidates, baseline = expanded_pool(m1, history, dense, max_candidates)
        pools[group] = {
            "candidate_item_ids": candidates,
            "m6_baseline_item_ids": baseline,
            "query_text": str(query.search_query),
            "search_infm_params_text": str(query.search_infm_params_text),
        }
        rank_maps = {"m1": rank_lookup(m1), "bm25": rank_lookup(bm25), "char": rank_lookup(char), "history": rank_lookup(history), "dense": rank_lookup(dense)}
        query_tokens = token_set(query.search_query)
        for pool_rank, item_id in enumerate(candidates, start=1):
            item = item_feature_map[item_id]
            title_tokens = token_set(item.get("item_title_raw", ""))
            params_tokens = token_set(item.get("item_infm_params_text", ""))
            title_union = len(query_tokens | title_tokens)
            params_union = len(query_tokens | params_tokens)
            row = {
                "query_group": group,
                "item_id": item_id,
                "pool_rank": float(pool_rank),
                "query_char_len": float(len(str(query.search_query))),
                "query_token_count": float(len(query_tokens)),
                "has_search_filters": float(bool(str(query.search_infm_params_text).strip())),
                "title_char_len": float(len(str(item.get("item_title_raw", "")))),
                "description_char_len": float(len(str(item.get("item_description_raw", "")))),
                "title_token_overlap": float(len(query_tokens & title_tokens)),
                "title_token_jaccard": float(len(query_tokens & title_tokens) / title_union) if title_union else 0.0,
                "params_token_overlap": float(len(query_tokens & params_tokens)),
                "params_token_jaccard": float(len(query_tokens & params_tokens) / params_union) if params_union else 0.0,
                "log_price": float(np.log1p(max(safe_float(item.get("item_price")), 0.0))),
                "item_rating": safe_float(item.get("item_rating")),
                "log_reviews": float(np.log1p(max(safe_float(item.get("item_rating_reviews_count")), 0.0))),
                "location_exact": float(str(item.get("item_location_id")) == str(query.search_location_id)),
                "phone_hidden": safe_float(item.get("item_is_phone_hidden")),
                "message_forbidden": safe_float(item.get("item_is_message_forbidden")),
            }
            for source_name, source_ranks in rank_maps.items():
                rank = source_ranks.get(item_id, SOURCE_TOP_K + 1)
                row[f"{source_name}_rank"] = float(rank)
                row[f"{source_name}_rr"] = 1.0 / (RRF_K + rank) if item_id in source_ranks else 0.0
                row[f"in_{source_name}"] = float(item_id in source_ranks)
            if labels_by_group is not None:
                row["label"] = float(item_id in labels_by_group.get(group, frozenset()))
            rows.append(row)
    return pd.DataFrame(rows), pools


train_gold_by_group = proxy_train_pairs.groupby("query_group", sort=False)["item_id"].agg(
    lambda values: frozenset(values.astype(str))
).to_dict()
train_frame_full, train_pools = build_feature_rows(
    proxy_train_queries, labels_by_group=train_gold_by_group, max_candidates=MAX_POOL_CANDIDATES,
)
valid_frame, valid_pools = build_feature_rows(
    validation_queries, labels_by_group=gold_by_group, max_candidates=MAX_POOL_CANDIDATES,
)

# Limit only training negatives while preserving selected positives and the protected baseline.
keep_indices: list[int] = []
for group, part in train_frame_full.groupby("query_group", sort=False):
    protected = set(train_pools[int(group)]["m6_baseline_item_ids"])
    keep = part.loc[(part["label"] > 0) | part["item_id"].isin(protected)].copy()
    if len(keep) < MAX_TRAIN_CANDIDATES:
        remainder = part.loc[~part.index.isin(keep.index)].head(MAX_TRAIN_CANDIDATES - len(keep))
        keep = pd.concat([keep, remainder], ignore_index=False)
    keep_indices.extend(keep.index.tolist())
train_frame = train_frame_full.loc[sorted(set(keep_indices))].copy()

train_group_coverage = train_frame_full.groupby("query_group")["label"].max()
valid_group_coverage = valid_frame.groupby("query_group")["label"].max()
print({
    "train_rows_full": len(train_frame_full),
    "train_rows_after_negative_cap": len(train_frame),
    "validation_rows": len(valid_frame),
    "train_candidate_coverage": round(float(train_group_coverage.mean()), 6),
    "validation_candidate_coverage": round(float(valid_group_coverage.mean()), 6),
})

# Groups without any candidate positive cannot create a ranking pair; exclude only from fitting.
train_groups_with_positive = set(train_group_coverage.index[train_group_coverage > 0])
ranker_train = train_frame.loc[train_frame["query_group"].isin(train_groups_with_positive)].sort_values(
    ["query_group", "pool_rank", "item_id"], kind="stable"
).reset_index(drop=True)
feature_columns = [
    column for column in ranker_train.columns
    if column not in {"query_group", "item_id", "label"}
]
assert feature_columns and ranker_train["label"].sum() > 0

train_pool = Pool(
    ranker_train[feature_columns].astype(np.float32),
    label=ranker_train["label"].astype(np.float32),
    group_id=ranker_train["query_group"].astype(np.int64),
)
catboost_params = {
    "loss_function": "YetiRank",
    "eval_metric": "NDCG:top=50",
    "iterations": 500,
    "depth": 8,
    "learning_rate": 0.08,
    "random_seed": SEED,
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "verbose": 100,
    "allow_writing_files": False,
    "thread_count": -1,
}
if USE_CATBOOST_GPU:
    catboost_params.update({"task_type": "GPU", "devices": "0"})

selector_started = time.perf_counter()
selector = CatBoostRanker(**catboost_params)
# No validation set / early stopping: frozen proxy stays evaluation-only.
selector.fit(train_pool)
selector_seconds = time.perf_counter() - selector_started

valid_features = valid_frame[feature_columns].astype(np.float32)
valid_frame["catboost_score"] = selector.predict(valid_features)
valid_frame = valid_frame.sort_values(["query_group", "catboost_score", "pool_rank", "item_id"], ascending=[True, False, True, True], kind="stable")
selector_rankings = [
    part.head(FINAL_K)["item_id"].astype(str).tolist()
    for _, part in valid_frame.groupby("query_group", sort=True)
]
baseline_rankings = [valid_pools[int(group)]["m6_baseline_item_ids"] for group in validation_queries["query_group"]]

def macro_recall(rankings: list[list[str]], gold: list[frozenset[str]], k: int = FINAL_K) -> float:
    return float(np.mean([
        len(set(prediction[:k]) & relevant) / len(relevant)
        for prediction, relevant in zip(rankings, gold, strict=True)
    ]))

baseline_recall = macro_recall(baseline_rankings, gold_sets)
selector_recall = macro_recall(selector_rankings, gold_sets)
candidate_oracle = macro_recall([
    valid_pools[int(group)]["candidate_item_ids"] for group in validation_queries["query_group"]
], gold_sets, MAX_POOL_CANDIDATES)
results_frame = pd.DataFrame([
    {"method": "M6_fixed_baseline", "recall@50": baseline_recall},
    {"method": "M7_CatBoost_selector", "recall@50": selector_recall},
    {"method": f"candidate_pool_oracle@{MAX_POOL_CANDIDATES}", "recall@50": candidate_oracle},
])
results_frame["gain_vs_m6"] = results_frame["recall@50"] - baseline_recall
display(results_frame)
print({
    "selector_seconds": round(selector_seconds, 2),
    "m6_recall@50": round(baseline_recall, 6),
    "catboost_recall@50": round(selector_recall, 6),
    "gain": round(selector_recall - baseline_recall, 6),
    "accept_selector": bool(selector_recall > baseline_recall),
})

## 7. Артефакты, ClearML и ZIP

In [ ]:
from shutil import make_archive
from IPython.display import FileLink, display

results_path = OUTPUT_DIR / "m7_e12_results.csv"
metrics_path = OUTPUT_DIR / "m7_e12_metrics.json"
model_path = OUTPUT_DIR / "catboost_selector.cbm"
importance_path = OUTPUT_DIR / "feature_importance.csv"
pool_path = OUTPUT_DIR / "validation_candidate_pool.jsonl"
scored_path = OUTPUT_DIR / "validation_scored_candidates.parquet"
manifest_path = OUTPUT_DIR / "m7_e12_manifest.json"

results_frame.to_csv(results_path, index=False)
selector.save_model(str(model_path))
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": selector.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
importance.to_csv(importance_path, index=False)
valid_frame.to_parquet(scored_path, index=False)
with pool_path.open("w", encoding="utf-8") as handle:
    for query in validation_queries.itertuples(index=False):
        group = int(query.query_group)
        payload = {
            "query_group": group,
            "search_query": str(query.search_query),
            "search_infm_params_text": str(query.search_infm_params_text),
            "candidate_item_ids": valid_pools[group]["candidate_item_ids"],
            "m6_baseline_item_ids": valid_pools[group]["m6_baseline_item_ids"],
        }
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

metrics = {
    "m6_recall@50": baseline_recall,
    "catboost_recall@50": selector_recall,
    "gain_vs_m6": selector_recall - baseline_recall,
    "candidate_pool_oracle": candidate_oracle,
    "accept_selector": bool(selector_recall > baseline_recall),
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
manifest = {
    "stage": "M7_E12_catboost_candidate_selector",
    "validation_protocol": "frozen_M0_proxy__group_disjoint__OOF_history",
    "seed": SEED,
    "candidate_sources": ["M1_RRF", "M2_OOF_nearest_history", "multilingual_E5"],
    "candidate_pool_limit": MAX_POOL_CANDIDATES,
    "train_negative_limit": MAX_TRAIN_CANDIDATES,
    "oof_folds": OOF_FOLDS,
    "feature_columns": feature_columns,
    "catboost_params": catboost_params,
    "metrics": metrics,
    "timing_seconds": {"selector_fit": selector_seconds, "notebook_total": time.perf_counter() - NOTEBOOK_STARTED},
    "dependencies": {name: importlib.metadata.version(name) for name in ("catboost", "clearml", "sentence-transformers", "torch", "pandas", "scikit-learn")},
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

for _, row in results_frame.iterrows():
    clearml_logger.report_scalar("Recall@50", str(row["method"]), float(row["recall@50"]), 0)
clearml_logger.report_table("M7 selector results", "summary", 0, table_plot=results_frame)
clearml_logger.report_table("M7 feature importance", "features", 0, table_plot=importance)
clearml_task.set_parameter("results/catboost_recall_at_50", selector_recall)
clearml_task.set_parameter("results/gain_vs_m6", selector_recall - baseline_recall)
for artifact_name, artifact_path in {
    "m7_results": results_path,
    "m7_metrics": metrics_path,
    "m7_manifest": manifest_path,
    "m7_feature_importance": importance_path,
    "m7_validation_candidate_pool": pool_path,
    "m7_catboost_model": model_path,
}.items():
    clearml_task.upload_artifact(artifact_name, artifact_object=artifact_path)

zip_path = OUTPUT_DIR.with_suffix(".zip")
make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR.parent), base_dir=OUTPUT_DIR.name)
clearml_task.upload_artifact("m7_e12_artifacts_zip", artifact_object=zip_path)
clearml_task.close()

assert all(len(row) <= FINAL_K and len(row) == len(set(row)) for row in selector_rankings)
print({"zip_path": str(zip_path), "candidate_pool_for_e13": str(pool_path), **metrics})
display(FileLink(zip_path))

## Решение после E12

Включать selector в M9 можно только при положительном `gain_vs_m6` на
frozen proxy. Иначе M9 остаётся на M6 quota policy. ZIP содержит
`catboost_selector.cbm`, feature importance и полный candidate cache, поэтому
результат воспроизводим и пригоден как вход для E13.
